# Diffusion Transformers & Rectified Flow Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: A DiT block with AdaLN

In [ ]:
```python

import torch

import torch.nn as nn

class AdaLNZero(nn.Module):

    """

    Adaptive LayerNorm with a gate. Predicts (scale, shift, gate) from the conditioning.

    Init such that the whole block starts as identity ("zero init").

    """

    def __init__(self, dim, cond_dim):

        super().__init__()

        self.norm = nn.LayerNorm(dim, elementwise_affine=False)

        self.mlp = nn.Linear(cond_dim, dim * 3)

        nn.init.zeros_(self.mlp.weight)

        nn.init.zeros_(self.mlp.bias)

    def forward(self, x, cond):

        scale, shift, gate = self.mlp(cond).chunk(3, dim=-1)

        h = self.norm(x) * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

        return h, gate.unsqueeze(1)

class DiTBlock(nn.Module):

    def __init__(self, dim=192, heads=3, mlp_ratio=4, cond_dim=192):

        super().__init__()

        self.adaln1 = AdaLNZero(dim, cond_dim)

        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)

        self.adaln2 = AdaLNZero(dim, cond_dim)

        self.mlp = nn.Sequential(

            nn.Linear(dim, dim * mlp_ratio),

            nn.GELU(),

            nn.Linear(dim * mlp_ratio, dim),

        )

    def forward(self, x, cond):

        h, gate1 = self.adaln1(x, cond)

        a, _ = self.attn(h, h, h, need_weights=False)

        x = x + gate1 * a

        h, gate2 = self.adaln2(x, cond)

        x = x + gate2 * self.mlp(h)

        return x

In [ ]:
```

`AdaLNZero` starts as an identity mapping because its MLP weights are initialised to zero. Training nudges the block away from identity; this stabilises deep transformer diffusion models dramatically.

### Step 2: A tiny DiT

In [ ]:
```python

def timestep_embedding(t, dim):

    import math

    half = dim // 2

    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)

    args = t[:, None].float() * freqs[None]

    return torch.cat([args.sin(), args.cos()], dim=-1)

class TinyDiT(nn.Module):

    def __init__(self, image_size=16, patch_size=2, in_channels=3, dim=96, depth=4, heads=3):

        super().__init__()

        self.patch_size = patch_size

        self.num_patches = (image_size // patch_size) ** 2

        self.patch = nn.Conv2d(in_channels, dim, kernel_size=patch_size, stride=patch_size)

        self.pos = nn.Parameter(torch.zeros(1, self.num_patches, dim))

        self.time_mlp = nn.Sequential(

            nn.Linear(dim, dim * 2),

            nn.SiLU(),

            nn.Linear(dim * 2, dim),

        )

        self.blocks = nn.ModuleList([DiTBlock(dim, heads, cond_dim=dim) for _ in range(depth)])

        self.norm_out = nn.LayerNorm(dim, elementwise_affine=False)

        self.head = nn.Linear(dim, patch_size * patch_size * in_channels)

    def forward(self, x, t):

        n = x.size(0)

        x = self.patch(x)

        x = x.flatten(2).transpose(1, 2) + self.pos

        t_emb = self.time_mlp(timestep_embedding(t, self.pos.size(-1)))

        for blk in self.blocks:

            x = blk(x, t_emb)

        x = self.norm_out(x)

        x = self.head(x)

        return self._unpatchify(x, n)

    def _unpatchify(self, x, n):

        p = self.patch_size

        h = w = int(self.num_patches ** 0.5)

        x = x.view(n, h, w, p, p, -1).permute(0, 5, 1, 3, 2, 4).reshape(n, -1, h * p, w * p)

        return x

In [ ]:
```

### Step 3: Rectified flow training

In [ ]:
```python

import torch.nn.functional as F

def rectified_flow_train_step(model, x0, optimizer, device):

    model.train()

    x0 = x0.to(device)

    n = x0.size(0)

    t = torch.rand(n, device=device)

    epsilon = torch.randn_like(x0)

    x_t = (1 - t[:, None, None, None]) * x0 + t[:, None, None, None] * epsilon

    target_velocity = epsilon - x0

    pred_velocity = model(x_t, t)

    loss = F.mse_loss(pred_velocity, target_velocity)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    return loss.item()

In [ ]:
```

Compare with DDPM's noise-prediction loss (Lesson 10): same structure, different target. Instead of predicting the noise `epsilon`, we predict the **velocity** `epsilon - x_0`, which points from data to noise along the straight-line interpolation.

### Step 4: Euler sampler

Rectified flow is an ODE. Euler's method is the simplest and, for a well-trained rectified-flow model, nearly as accurate as higher-order solvers at 20+ steps.

In [ ]:
```python

@torch.no_grad()

def rectified_flow_sample(model, shape, steps=20, device="cpu"):

    model.eval()

    x = torch.randn(shape, device=device)

    dt = 1.0 / steps

    t = torch.ones(shape[0], device=device)

    for _ in range(steps):

        v = model(x, t)

        x = x - dt * v

        t = t - dt

    return x

In [ ]:
```

20 steps. On a trained model this produces samples comparable to 1000-step DDPM.

### Step 5: End-to-end smoke test

In [ ]:
```python

import numpy as np

def synthetic_blobs(num=200, size=16, seed=0):

    rng = np.random.default_rng(seed)

    out = np.zeros((num, 3, size, size), dtype=np.float32)

    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")

    for i in range(num):

        cx, cy = rng.uniform(4, size - 4, size=2)

        r = rng.uniform(2, 4)

        mask = (xx - cx) ** 2 + (yy - cy) ** 2 < r ** 2

        colour = rng.uniform(-1, 1, size=3)

        for c in range(3):

            out[i, c][mask] = colour[c]

    return torch.from_numpy(out)

In [ ]:
```

Train a `TinyDiT` on this with rectified flow. After 500 steps, sampled outputs should look like faint blobs of colour.

## Exercises

In [ ]:
1. **(Easy)** Train the TinyDiT above on the synthetic blob dataset for 500 steps. Compare samples produced with 10, 20, and 50 Euler steps.
2. **(Medium)** Add text conditioning by concatenating a learned class embedding to the time embedding (10 blob "classes" by colour). Sample with class 0, 5, and 9 and verify colours match.
3. **(Hard)** Compute the Fréchet distance (FID proxy) between generated samples from rectified-flow and DDPM versions of the same-size network trained on the same data for the same number of steps. Report which converges faster.